# IM 主力分钟拼接

Hydra + MLflow 研究的公共输入准备。沿用 `draft_experiments_v2/01_01_im_main_continuous.ipynb` 的选约、换月价差和双端点 log 复权逻辑。

使用 `latitude` 内核，从上到下逐格运行。数据库与正式日历只读；输入完整性检查在 Notebook 外执行，不加入重复防御代码。

前一交易日提供选约信号，只向更晚到期合约切换，成交量门槛为严格超过 1.10 倍。双端点复权依赖完整快照；后续季节性使用原始 `open/close`。

输出：本项目 `02_data/derived/im_main_continuous_1m.parquet`。

### 1. 导入依赖与设置路径

In [ ]:
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "02_Futures_Lakehouse"))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings
from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
)

from notebook_schema_browser import display_schema_metadata

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

strategy_dir = candidate_root / "01_project_collection/JQ_strategy"
research_dir = strategy_dir / "intraday_modeling_research"
output_dir = research_dir / "02_data/derived"
database_path = (
    strategy_dir / "financial_futures_data/data/warehouse/financial_futures.duckdb"
)
output_path = output_dir / "im_main_continuous_1m.parquet"

### 2. 查看日历契约与设置读取字段

In [ ]:
display_schema_metadata([FUTURES_CONTRACT_CALENDAR_SCHEMA])

calendar_table_name = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"table_name"].decode("utf-8")
)
calendar_primary_key = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"primary_key"]
    .decode("utf-8").split(",")
)
calendar_partition_columns = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"partition_columns"]
    .decode("utf-8").split(",")
)
calendar_dir = settings.futures_lake_root / "silver" / calendar_table_name

calendar_columns = [
    "contract_code", "trading_date", "list_date", "delist_date",
    "session_number", "session_start_at", "session_end_at", "minute_count",
]
calendar_read_schema = pa.schema([
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column)
    for column in calendar_columns
])
calendar_partitioning = ds.partitioning(
    pa.schema([
        FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column)
        for column in calendar_partition_columns
    ]),
    flavor="hive",
)

### 3. 只读 IM 日线

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    daily_table = connection.execute("""
        SELECT contract_code, trading_date, close, volume, open_interest
        FROM futures_daily
        WHERE contract_code LIKE 'IM%.CCFX'
        ORDER BY trading_date, contract_code
    """).to_arrow_table()

daily_df = daily_table.to_pandas(types_mapper=pd.ArrowDtype)

start_date = daily_df["trading_date"].min()
end_date = daily_df["trading_date"].max()

print(f"日线范围：{start_date} 至 {end_date}；共 {len(daily_df):,} 条")

### 4. 读取合约日历

In [ ]:
calendar_dataset = ds.dataset(
    calendar_dir,
    format="parquet",
    partitioning=calendar_partitioning,
)
calendar_table = calendar_dataset.to_table(
    columns=calendar_columns,
    filter=(
        (ds.field("exchange_code") == "CCFX")
        & (ds.field("underlying_code") == "IM")
        & (ds.field("trading_date") >= start_date)
        & (ds.field("trading_date") <= end_date)
    ),
)
calendar_df = calendar_table.cast(calendar_read_schema).to_pandas(types_mapper=pd.ArrowDtype)
calendar_df = calendar_df.sort_values(calendar_primary_key).reset_index(drop=True)

contract_df = (
    calendar_df[["contract_code", "list_date", "delist_date"]]
    .drop_duplicates()
    .sort_values("contract_code")
    .reset_index(drop=True)
)
trading_dates = sorted(calendar_df["trading_date"].unique().tolist())

### 5. 按前一交易日信号选择主力

In [ ]:
roll_volume_ratio = 1.10
current_contract_code = None
current_delist_date = None
main_mapping_rows = []

for signal_date, trading_date in zip(trading_dates[:-1], trading_dates[1:]):
    # 信号日已上市，且执行日仍未到期的合约。
    candidate_df = contract_df.loc[
        (contract_df["list_date"] <= signal_date)
        & (contract_df["delist_date"] >= trading_date)
    ].merge(
        daily_df.loc[
            daily_df["trading_date"] == signal_date,
            ["contract_code", "volume", "open_interest"],
        ],
        on="contract_code",
        how="left",
    )

    # 成交量并列时，依次比较持仓量、到期日和合约代码。
    candidate_df = candidate_df.loc[candidate_df["volume"] > 0].sort_values(
        ["volume", "open_interest", "delist_date", "contract_code"],
        ascending=[False, False, True, True],
    )

    if current_contract_code is None:
        selected_contract = candidate_df.iloc[0]
        selection_reason = "初始选择"
    else:
        current_candidate_df = candidate_df.loc[
            candidate_df["contract_code"] == current_contract_code
        ]
        later_candidate_df = candidate_df.loc[
            candidate_df["delist_date"] > current_delist_date
        ]

        if current_candidate_df.empty:
            selected_contract = later_candidate_df.iloc[0]
            selection_reason = (
                "到期换月" if current_delist_date < trading_date else "零成交量换月"
            )
        elif (
            not later_candidate_df.empty
            and later_candidate_df.iloc[0]["volume"]
            > roll_volume_ratio * current_candidate_df.iloc[0]["volume"]
        ):
            selected_contract = later_candidate_df.iloc[0]
            selection_reason = "成交量触发换月"
        else:
            selected_contract = current_candidate_df.iloc[0]
            selection_reason = "维持主力"

    selected_contract_code = selected_contract["contract_code"]
    main_mapping_rows.append({
        "trading_date": trading_date,
        "signal_trading_date": signal_date,
        "contract_code": selected_contract_code,
        "previous_contract_code": current_contract_code,
        "is_roll_day": (
            current_contract_code is not None
            and selected_contract_code != current_contract_code
        ),
        "selected_signal_volume": float(selected_contract["volume"]),
        "selection_reason": selection_reason,
    })
    current_contract_code = selected_contract_code
    current_delist_date = selected_contract["delist_date"]

main_mapping_df = pd.DataFrame(main_mapping_rows)

roll_mapping_df = main_mapping_df.loc[
    main_mapping_df["is_roll_day"]
].reset_index(drop=True)

print(f"主力覆盖 {len(main_mapping_df)} 个交易日；换月 {len(roll_mapping_df)} 次")
main_mapping_df.head()

### 6. 计算换月 log 价差

In [ ]:
close_by_date_df = daily_df.pivot(
    index="trading_date",
    columns="contract_code",
    values="close",
).sort_index()

roll_gap_rows = []

for roll in roll_mapping_df.itertuples(index=False):
    common_close_df = close_by_date_df.loc[
        close_by_date_df.index <= roll.signal_trading_date,
        [roll.previous_contract_code, roll.contract_code],
    ].dropna()

    common_close_df = common_close_df.loc[
        (common_close_df > 0).all(axis=1)
    ]

    reference_close = common_close_df.iloc[-1]
    old_reference_close = float(reference_close[roll.previous_contract_code])
    new_reference_close = float(reference_close[roll.contract_code])
    roll_log_gap = np.log(new_reference_close) - np.log(old_reference_close)

    roll_gap_rows.append((
        roll.trading_date,
        roll.signal_trading_date,
        reference_close.name,
        roll.previous_contract_code,
        roll.contract_code,
        old_reference_close,
        new_reference_close,
        roll_log_gap,
    ))

roll_gap_df = pd.DataFrame(
    roll_gap_rows,
    columns=[
        "trading_date", "signal_trading_date", "roll_reference_trading_date",
        "previous_contract_code", "contract_code",
        "old_reference_close", "new_reference_close", "roll_log_gap",
    ],
)

roll_gap_df

### 7. 累计换月价差

In [ ]:
roll_log_gap_by_date = roll_gap_df.set_index("trading_date")["roll_log_gap"]

main_mapping_df["roll_log_gap"] = (
    main_mapping_df["trading_date"]
    .map(roll_log_gap_by_date)
    .astype("float64")
)

main_mapping_df["roll_log_gap"] = main_mapping_df["roll_log_gap"].fillna(0.0)
main_mapping_df["cumulative_roll_log_gap"] = (
    main_mapping_df["roll_log_gap"].cumsum()
)

### 8. 按每日主力映射读取分钟行情

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    connection.register("main_mapping", main_mapping_df)

    minute_table = connection.execute("""
        SELECT
            f.contract_code, f.trading_date, f.session_number, f.bar_at,
            f.open, f.high, f.low, f.close,
            f.volume, f.money, f.open_interest,
            m.signal_trading_date, m.is_roll_day, m.cumulative_roll_log_gap
        FROM futures_minute AS f
        JOIN main_mapping AS m
          ON f.contract_code = m.contract_code
         AND f.trading_date = m.trading_date
        ORDER BY f.bar_at
    """).to_arrow_table()

minute_df = minute_table.to_pandas(types_mapper=pd.ArrowDtype)

### 9. 双端点 log 复权

In [ ]:
price_columns = ["open", "high", "low", "close"]
minute_count = len(minute_df)

cumulative_roll_log_gap = minute_df[
    "cumulative_roll_log_gap"
].to_numpy(dtype=np.float64)

relative_roll_log_gap = (
    cumulative_roll_log_gap - cumulative_roll_log_gap[0]
)
total_roll_log_gap = float(relative_roll_log_gap[-1])

# 按实际交易分钟分配权重，首分钟为 0，末分钟为 1。
adjustment_weight = np.linspace(0.0, 1.0, minute_count)
log_adjustment = (
    -relative_roll_log_gap + adjustment_weight * total_roll_log_gap
)
price_multiplier = np.exp(log_adjustment)

minute_df["bidirectional_adjustment_weight"] = adjustment_weight
minute_df["bidirectional_log_adjustment"] = log_adjustment

# 复权依赖完整快照，记录其末分钟时刻。
minute_df["price_adjustment_available_at"] = minute_df["bar_at"].iloc[-1]

for price_column in price_columns:
    raw_price_values = minute_df[price_column].to_numpy(dtype=np.float64)

    minute_df[f"bidirectional_adjusted_log_{price_column}"] = (
        np.log(raw_price_values) + log_adjustment
    )
    minute_df[f"bidirectional_adjusted_{price_column}"] = (
        raw_price_values * price_multiplier
    )

minute_df.head()

### 10. 设置输出类型并保存 Parquet

In [ ]:
minute_schema = pa.schema([
    pa.field("contract_code", pa.string(), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False),
    pa.field("session_number", pa.int8(), nullable=False),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),

    *[
        pa.field(column, pa.float64(), nullable=False)
        for column in price_columns + ["volume", "money", "open_interest"]
    ],

    pa.field("signal_trading_date", pa.date32(), nullable=False),
    pa.field("is_roll_day", pa.bool_(), nullable=False),
    pa.field("cumulative_roll_log_gap", pa.float64(), nullable=False),
    pa.field("bidirectional_adjustment_weight", pa.float64(), nullable=False),
    pa.field("bidirectional_log_adjustment", pa.float64(), nullable=False),
    pa.field(
        "price_adjustment_available_at",
        pa.timestamp("us", tz="Asia/Shanghai"),
        nullable=False,
    ),

    *[
        pa.field(f"bidirectional_adjusted_log_{column}", pa.float64(), nullable=False)
        for column in price_columns
    ],
    *[
        pa.field(f"bidirectional_adjusted_{column}", pa.float64(), nullable=False)
        for column in price_columns
    ],
], metadata={
    b"price_adjustment_method": b"dual endpoint log adjustment",
    b"price_adjustment_formula": b"a_i=-(G_i-G_0)+i/(n-1)*(G_last-G_0)",
    b"price_adjustment_scope": b"whole input snapshot; retrospective",
    b"total_roll_log_gap": repr(total_roll_log_gap).encode("utf-8"),
})

minute_table = pa.Table.from_pandas(
    minute_df[minute_schema.names],
    schema=minute_schema,
    preserve_index=False,
)

output_dir.mkdir(parents=True, exist_ok=True)
temporary_output_path = output_path.with_suffix(".parquet.tmp")
pq.write_table(minute_table, temporary_output_path, compression="zstd")
temporary_output_path.replace(output_path)


print(f"已保存 {len(minute_df):,} 条分钟数据：{output_path}")